# Biohub Cell Tracking Baseline - Colab

Bu notebook Colab + Google Drive kullanimi icin hazirlandi. Dataset henuz inmediyse hucreler fonksiyonlari yukler, ama tam pipeline'i baslatmaz.

Beklenen Drive proje yapisi:

`/content/drive/MyDrive/Biohub - Cell Tracking During Development/`

Dataset bu proje klasorunun altindaki `data/` klasorunde aranir. Zip dogrudan `data/` icine acilmis olabilir veya `data/biohub-cell-tracking-during-development/` altinda olabilir.

Bu notebook, ayni klasorde bulunan `biohub_baseline.py` dosyasini kullanir.

## 1. Google Drive'i Bagla

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Gerekli Paketleri Kur

Colab'de paketlerin bir kismi zaten kurulu olabilir. Bu hucre eksik olanlari tamamlar.

In [ ]:
!pip install -q zarr numcodecs scipy scikit-image matplotlib tqdm pandas numpy

## 3. Proje Kodunu Yukle

Bu notebook ile `biohub_baseline.py` ayni klasorde olursa direkt import edilir. Colab'e sadece notebook'u yuklediysen, sol panelden `biohub_baseline.py` dosyasini da yukle veya Drive'daki proje klasorune koy.

In [ ]:
from pathlib import Path
import sys

PROJECT_DIR = Path('/content')
DRIVE_PROJECT_DIR = Path('/content/drive/MyDrive/Biohub - Cell Tracking During Development')
DRIVE_DATA_DIR = DRIVE_PROJECT_DIR / 'data'

for candidate in [DRIVE_PROJECT_DIR, PROJECT_DIR, Path.cwd()]:
    if (candidate / 'biohub_baseline.py').exists():
        sys.path.insert(0, str(candidate))
        print(f'Using project code from: {candidate}')
        break
else:
    raise FileNotFoundError(
        'biohub_baseline.py bulunamadi. Notebook ile ayni Colab klasorune yukle '
        'veya Drive proje klasorune koy.'
    )

import biohub_baseline as bh
print('biohub_baseline imported successfully')

## 4. Dataset Yolunu Ayarla ve Kontrol Et

Dataset henuz inmiyse bu hucre hata vermez; aciklayici mesaj basar ve durur.

In [ ]:
bh.find_dataset_base_dir('/content/drive/MyDrive/Biohub - Cell Tracking During Development/data')

print('BASE_DIR =', bh.BASE_DIR)
available = bh.dataset_available(bh.BASE_DIR)
print('dataset_available =', available)

## 5. Dataset'i Incele

Download/extraction bittikten sonra calistir.

In [ ]:
summary = bh.inspect_dataset(bh.BASE_DIR)

## 6. Sample Submission Formatini Incele

In [ ]:
sample_submission_df = bh.inspect_sample_submission(bh.SAMPLE_SUBMISSION_PATH)

## 7. Parametreleri Ayarla

Baslangicta guvenli ve hizli deneme icin `max_detections_per_frame` siniri koyabilirsin.

In [ ]:
bh.DEBUG_MODE = False
bh.RUN_TEST_SUBMISSION = False

bh.DETECTION_PARAMS.update({
    'percentile_low': 1,
    'percentile_high': 99.8,
    'gaussian_sigma': 1.0,
    'min_distance': 3,
    'threshold_abs': 0.2,
    'max_detections_per_frame': None,
})

bh.LINKING_PARAMS.update({
    'z_scale': 1.625,
    'y_scale': 0.40625,
    'x_scale': 0.40625,
    'max_distance_um': 7.0,
})

bh.DETECTION_PARAMS, bh.LINKING_PARAMS

## 8. Debug Modu

Tek train sample uzerinde kucuk bir deneme yapar. Tum pipeline'i calistirmadan once bunu kullan.

In [ ]:
if bh.dataset_available(bh.BASE_DIR):
    bh.run_debug_mode(bh.BASE_DIR)
else:
    print('Dataset hazir degil; debug modu baslatilmadi.')

## 9. Test Submission Modu

Bu hucre test setinin tamaminda calisir. Dataset buyuk oldugu icin once debug modunu tamamla.

In [ ]:
RUN_FULL_TEST_SUBMISSION_NOW = False

if RUN_FULL_TEST_SUBMISSION_NOW:
    if bh.dataset_available(bh.BASE_DIR):
        bh.run_test_submission_mode(bh.BASE_DIR)
    else:
        print('Dataset hazir degil; submission pipeline baslatilmadi.')
else:
    print('RUN_FULL_TEST_SUBMISSION_NOW False. Tam test pipeline baslatilmadi.')